# Generating and Explaining Recommendations

In this notebook, we move past aggregate metrics and look at individual users. We will generate Top-10 recommendations for a diverse set of users, compare model outputs side-by-side, and explore how to explain these recommendations to the end user.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import config
from src.data_loader import load_processed, load_movie_titles
from src.preprocessing import sample_data, create_train_test_split
from src.models.svd_model import SVDRecommender
from src.models.knn_cf import UserBasedCF
from src.models.neural_cf import NeuralCFRecommender
from src.recommender import RecommendationEngine

# Load data
df = load_processed(os.path.join('..', config.PROCESSED_DATA_DIR, 'all_ratings.parquet'))
titles = load_movie_titles(os.path.join('..', config.RAW_DATA_DIR, 'movie_titles.txt'))

sampled_df = sample_data(df)
train_df, test_df = create_train_test_split(sampled_df, strategy='random')

Let's train a quick SVD model to power our primary Recommendation Engine. We'll use this as our 'best' model based on the previous notebook's findings.

In [ ]:
from src.preprocessing import build_surprise_dataset

trainset_surp = build_surprise_dataset(train_df).build_full_trainset()
svd_model = SVDRecommender()
svd_model.fit(trainset_surp)

rec_engine = RecommendationEngine(svd_model, titles)

## Selecting Diverse Users

To thoroughly test our engine, we shouldn't just look at random users. We want to see how the model behaves for different user personas: a power user, a casual viewer, someone with niche tastes, etc.

In [ ]:
user_counts = train_df.groupby('user_id')['rating'].count()

# Find some interesting users
power_user = user_counts[user_counts > 500].index[0] if len(user_counts[user_counts > 500]) > 0 else user_counts.idxmax()
casual_user = user_counts[(user_counts > 15) & (user_counts < 30)].index[0]
mainstream_user = user_counts[(user_counts > 100) & (user_counts < 200)].index[0]

diverse_users = [power_user, casual_user, mainstream_user]
user_labels = ['Power User', 'Casual User', 'Mainstream User']

print(f"Power User ID: {power_user} ({user_counts[power_user]} ratings)")
print(f"Casual User ID: {casual_user} ({user_counts[casual_user]} ratings)")
print(f"Mainstream User ID: {mainstream_user} ({user_counts[mainstream_user]} ratings)")

## Generating Recommendations for Personas

Let's look at their top-rated movies to understand their taste, and then see what the SVD model recommends.

In [ ]:
for u, label in zip(diverse_users, user_labels):
    print("="*75)
    print(f"{label.upper()} PROFILE (ID: {u})")
    print("="*75)
    
    hist = rec_engine.get_user_history(u, train_df, n=5)
    print("Recent/Top Favorites:")
    display(hist)
    
    rated_items = set(train_df[train_df['user_id'] == u]['movie_id'])
    recs = rec_engine.generate_top_k(u, k=10, exclude_rated=True, rated_items=rated_items)
    
    print("\n")
    rec_engine.display_recommendations(u, recs)
    print("\n")

## Comparing Model Outputs

Let's train dummy KNN and NCF models really quickly to compare their outputs against SVD for the exact same user.

In [ ]:
# Setup KNN
knn_model = UserBasedCF()
knn_model.fit(trainset_surp)

# Setup NCF
from sklearn.preprocessing import LabelEncoder
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
user_encoder.fit(sampled_df['user_id'])
item_encoder.fit(sampled_df['movie_id'])

train_ncf_df = train_df.copy()
train_ncf_df['user_idx'] = user_encoder.transform(train_df['user_id'])
train_ncf_df['item_idx'] = item_encoder.transform(train_df['movie_id'])

ncf_model = NeuralCFRecommender(len(user_encoder.classes_), len(item_encoder.classes_))
ncf_model.user_encoder = user_encoder
ncf_model.item_encoder = item_encoder
# Just 1 epoch for demo purposes
ncf_model.fit(train_ncf_df, epochs=1) 

print("Models trained. Ready to compare.")

In [ ]:
target_user = mainstream_user
all_items = titles.index.tolist()
user_rated = train_df[train_df['user_id'] == target_user]['movie_id'].tolist()

svd_top = svd_model.get_top_n(target_user, all_items, n=10, exclude_rated=True)
knn_top = knn_model.get_top_n(target_user, all_items, n=10, exclude_rated=True)
ncf_top = ncf_model.get_top_n(target_user, all_items, n=10, exclude_rated=True, user_rated_items=user_rated)

print(f"TOP 10 COMPARISON FOR USER {target_user}")
print("-" * 105)
print(f"{'SVD Output':<33} | {'KNN Output':<33} | {'NCF Output':<33}")
print("-" * 105)

for i in range(10):
    svd_t = titles.loc[svd_top[i][0], 'title'][:30] if i < len(svd_top) else '-'
    knn_t = titles.loc[knn_top[i][0], 'title'][:30] if i < len(knn_top) else '-'
    ncf_t = titles.loc[ncf_top[i][0], 'title'][:30] if i < len(ncf_top) else '-'
    
    print(f"{svd_t:<33} | {knn_t:<33} | {ncf_t:<33}")

## Explainability

Users trust recommendations more if they understand *why* they were suggested. Our `RecommendationEngine` provides a basic scaffolding for explanations based on the underlying algorithm.

In [ ]:
svd_engine = RecommendationEngine(svd_model, titles)
knn_engine = RecommendationEngine(knn_model, titles)
ncf_engine = RecommendationEngine(ncf_model, titles)

sample_movie_id = svd_top[0][0]

print(f"Target Movie: {titles.loc[sample_movie_id, 'title']}\n")

print("SVD Explanation:")
print(svd_engine.explain_recommendation(target_user, sample_movie_id))

print("\nKNN Explanation:")
print(knn_engine.explain_recommendation(target_user, sample_movie_id))

print("\nNeural CF Explanation:")
print(ncf_engine.explain_recommendation(target_user, sample_movie_id))

## Failure Analysis & Edge Cases

* **Popularity Bias:** Notice how often "Lord of the Rings" or "Shawshank Redemption" appear in the recommendations, especially for the Casual User? This is popularity bias. The models learn that these movies are safe bets. To fix this, we might need to penalize popular items or boost diversity in the recommendation engine.
* **Cold Start:** If a brand new user joins and rates zero movies, our models fail completely. SVD will return the global mean (e.g. 3.5 stars) for everything, and KNN will crash or return random neighbors. We need an onboarding flow (asking the user to select 5 movies they like) or a content-based fallback model to solve this.
* **Niche Bubble:** For users with very specific tastes, SVD sometimes struggles to pull them out of their bubble, occasionally recommending slightly off-genre movies simply because they share latent factors with highly-rated niche items.

## Manual Verification: MAP@10

To ensure we understand our evaluation metrics (from `src/evaluation.py`), let's manually trace through Mean Average Precision at 10 (MAP@10) for a single user.

In [ ]:
# Assume the user actually watched and liked (true rating >= 3.5) the following movies:
relevant_items = {101, 202, 303}

# Our model recommends these 10 items in order:
recommendations = [101, 999, 888, 202, 777, 666, 555, 303, 444, 111]

ap = 0.0
hits = 0

print("Tracing AP@10 Calculation:\n")
for i, rec in enumerate(recommendations):
    if rec in relevant_items:
        hits += 1
        precision_at_i = hits / (i + 1)
        ap += precision_at_i
        print(f"Rank {i+1}: Hit! Item {rec}. Precision@{i+1} = {hits}/{i+1} = {precision_at_i:.2f}")
    else:
        print(f"Rank {i+1}: Miss.")

# Final AP is the sum of precision at hits, divided by min(total_relevant, k)
final_ap = ap / min(len(relevant_items), 10)
print(f"\nTotal AP for user: {final_ap:.4f}")

# MAP@10 is simply the mean of this final_ap across all users in the test set.